# 🎧 EQ Personalizado con IA

Este proyecto usa inteligencia artificial para analizar cómo escuchas y generar una corrección de audio (EQ) personalizada para ti.

---

## ¿Cómo funciona?

```
PASO 1              PASO 2                    PASO 3
Cargar datos   →   Entrenar dos IAs      →   Hacerte la prueba de audición
de parlantes        IA 1: predice              y generar tus filtros EQ
y audífonos         qué tanto mejora           personalizados
                    el audio con EQ
                    IA 2: predice
                    los filtros exactos
```

---

## Archivos que necesitas tener subidos en Colab

| Archivo | Para qué se usa |
|---|---|
| `devices_wide.csv` | Datos de parlantes y audífonos con sus filtros EQ |
| `frequency_response.csv` | Curvas de respuesta de frecuencia de cada dispositivo |

---

## ⚠️ Importante antes de empezar
- Ejecuta las celdas **en orden, de arriba hacia abajo**
- Para ejecutar una celda: haz clic en ella y presiona **Shift + Enter**
- Para la prueba de audición: **usa audífonos** y sube el volumen al máximo

---
# PARTE 1 — Cargar los datos

Primero subimos los archivos CSV al entorno de Colab y los cargamos en memoria.

In [ ]:
# Subir los archivos desde tu computador a Colab
# Cuando ejecutes esta celda aparecerá un botón — úsalo para subir
# devices_wide.csv y frequency_response.csv

from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np

# Cargar el archivo principal (contiene los filtros EQ y los puntajes)
df_wide = pd.read_csv('devices_wide.csv')

# Cargar el archivo de curvas de frecuencia (cómo suena cada dispositivo)
df_freq = pd.read_csv('frequency_response.csv')

# Eliminar filas que no tienen puntaje (score_delta vacío)
df_wide = df_wide.dropna(subset=['score_delta'])

print(f'Dispositivos cargados: {df_wide["device_name"].nunique()}')
print(f'Tipos: {df_wide["device_type"].value_counts().to_dict()}')

---
# PARTE 2 — IA 1: Predictor de mejora de audio

**¿Qué hace esta IA?**
Recibe la curva de frecuencias de un dispositivo (cómo suena) y predice cuánto mejoraría su puntaje de calidad al aplicarle un EQ.

**¿Por qué sirve?**
Nos dice qué tan efectiva puede ser la corrección antes de calcularla. Si el score sube mucho, vale la pena aplicar EQ.

**Modelo usado: Ridge Regression**
Es una regresión lineal con control extra para manejar las 248 columnas de frecuencia sin confundirse. Simple, rápido y explicable.

In [ ]:
# Identificar las columnas de frecuencia en devices_wide
# Estas columnas representan qué tan fuerte suena cada banda de frecuencia
# Ejemplo: f_1000Hz = nivel de sonido a 1000 Hz en decibelios (dB)

freq_cols = [col for col in df_wide.columns
             if col.startswith('f') and not col.endswith('_type')
             and not col.endswith('_hz') and not col.endswith('_db')
             and not col.endswith('_q')]

# X = lo que el modelo recibe como entrada (las curvas de frecuencia)
# y = lo que el modelo debe predecir (cuánto mejora el puntaje con EQ)
X = df_wide[freq_cols].fillna(0)
y = df_wide['score_delta']

print(f'Entrada  X: {X.shape[0]} dispositivos × {X.shape[1]} frecuencias')
print(f'Objetivo y: {y.shape[0]} valores de mejora')
print(f'Mejora promedio con EQ: {y.mean():.2f} puntos')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Dividir datos: 80% para aprender, 20% para evaluar
# random_state=42 asegura que siempre se dividan igual
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar: poner todos los valores en la misma escala
# Sin esto, frecuencias altas (20000 Hz) dominarían sobre las bajas (20 Hz)
scaler_ia1 = StandardScaler()
X_train_scaled = scaler_ia1.fit_transform(X_train)
X_test_scaled  = scaler_ia1.transform(X_test)

# Crear y entrenar el modelo
# alpha=100 es el nivel de control — evita que el modelo se vuelva inestable
ia1 = Ridge(alpha=100)
ia1.fit(X_train_scaled, y_train)

print(f'IA 1 entrenada con {len(X_train)} dispositivos')

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

y_pred_ia1 = ia1.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred_ia1)
r2  = r2_score(y_test, y_pred_ia1)

print(f'Error promedio (MAE): {mae:.3f} puntos')
print(f'Precisión (R²):       {r2:.3f}  —  el modelo explica el {r2*100:.1f}% de los casos')
print()
if r2 > 0.7:
    print('✓ Buen modelo')
elif r2 > 0.4:
    print('~ Modelo aceptable')
else:
    print('✗ Modelo débil')

# Gráfica: puntos cerca de la línea roja = buenas predicciones
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_ia1, alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=1.5, label='Predicción perfecta')
plt.xlabel('Mejora real')
plt.ylabel('Mejora predicha')
plt.title('IA 1 — Predicciones vs. valores reales')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ver qué frecuencias tienen más influencia en la mejora
# Verde = esa frecuencia ayuda a subir el puntaje
# Rojo  = esa frecuencia baja el puntaje

coefs = pd.Series(ia1.coef_, index=freq_cols)
top20 = coefs.abs().nlargest(20).index
top20_coefs = coefs[top20].sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colores = ['#e74c3c' if v < 0 else '#2ecc71' for v in top20_coefs]
top20_coefs.plot(kind='barh', ax=ax, color=colores)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Influencia en la mejora del audio')
ax.set_title('Top 20 frecuencias más influyentes')
labels = [l.replace('f_', '').replace('Hz', ' Hz') for l in top20_coefs.index]
ax.set_yticklabels(labels, fontsize=9)
plt.tight_layout()
plt.show()

---
# PARTE 3 — IA 2: Generador de filtros EQ

**¿Qué hace esta IA?**
Recibe la curva de frecuencias de un dispositivo y genera los 7 filtros EQ necesarios para corregirlo.

Cada filtro tiene 3 valores:
- **fc_hz** — en qué frecuencia actúa el filtro (ej: 1000 Hz)
- **gain_db** — cuánto sube (+) o baja (-) el volumen en esa frecuencia
- **q** — qué tan ancho o estrecho es el ajuste

**Modelo usado: Random Forest**
Entrena 100 árboles de decisión en paralelo y promedia sus respuestas. Más preciso que Ridge para relaciones complejas entre frecuencias.

In [ ]:
# Para esta IA necesitamos combinar los dos archivos
# devices_wide tiene los filtros EQ, frequency_response tiene las curvas

freq_cols_ia2 = [col for col in df_freq.columns
                 if col.startswith('f_') and col.endswith('Hz')]

# Unir por nombre de dispositivo, tipo y variante de EQ
df = pd.merge(
    df_wide,
    df_freq[['device_name', 'device_type', 'eq_variant'] + freq_cols_ia2],
    on=['device_name', 'device_type', 'eq_variant'],
    how='inner'
)

# Las 21 salidas del modelo (7 filtros × 3 valores cada uno)
filtros_cols = ['f1_fc_hz','f1_gain_db','f1_q',
                'f2_fc_hz','f2_gain_db','f2_q',
                'f3_fc_hz','f3_gain_db','f3_q',
                'f4_fc_hz','f4_gain_db','f4_q',
                'f5_fc_hz','f5_gain_db','f5_q',
                'f6_fc_hz','f6_gain_db','f6_q',
                'f7_fc_hz','f7_gain_db','f7_q']

df = df.dropna(subset=filtros_cols + ['score_delta'])

X2 = df[freq_cols_ia2].fillna(0)
y2 = df[filtros_cols]

print(f'Dispositivos disponibles: {len(df)}')
print(f'Entrada  X: {X2.shape}')
print(f'Salida   y: {y2.shape}  →  21 valores por dispositivo')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

# Normalizar igual que en la IA 1
scaler_ia2 = StandardScaler()
X2_train_scaled = scaler_ia2.fit_transform(X2_train.values.astype(float))
X2_test_scaled  = scaler_ia2.transform(X2_test.values.astype(float))

# Random Forest con 50 árboles — balance entre velocidad y precisión
# n_jobs=-1 usa todos los núcleos del CPU para ir más rápido
ia2 = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
)
ia2.fit(X2_train_scaled, y2_train)

print(f'IA 2 entrenada con {len(X2_train)} dispositivos')
print('(Este paso puede tardar 3-5 minutos)')

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

y2_pred = ia2.predict(X2_test_scaled)

print(f'{'Filtro':<20}  {'Error MAE':>10}  {'Precisión R²':>12}')
print('-' * 48)
for i, col in enumerate(filtros_cols):
    mae = mean_absolute_error(y2_test.iloc[:, i], y2_pred[:, i])
    r2  = r2_score(y2_test.iloc[:, i], y2_pred[:, i])
    print(f'{col:<20}  {mae:>10.2f}  {r2:>12.3f}')

---
# PARTE 4 — Prueba de audición

Aquí es donde entra tu audición. El sistema reproduce tonos a diferentes frecuencias, subiendo el volumen gradualmente, y detecta en qué punto empiezas a escuchar cada uno.

**Ese punto se llama umbral auditivo** — mientras más alto el volumen necesario, mayor la dificultad auditiva en esa frecuencia.

**Instrucciones:**
1. Ponte los audífonos
2. Sube el volumen de tu dispositivo al máximo
3. Cuando escuches el tono presiona **Enter**
4. Si no lo escuchas escribe **n** y presiona Enter

In [ ]:
from IPython.display import Audio, display
import numpy as np
import time

def generar_tono(frecuencia, duracion=1.5, volumen=0.3, sample_rate=44100):
    """Genera un tono puro a la frecuencia indicada.
    El 'envelope' evita clicks molestos al inicio y al final del tono."""
    t = np.linspace(0, duracion, int(sample_rate * duracion))
    envelope = np.ones_like(t)
    fade = int(sample_rate * 0.05)
    envelope[:fade]  = np.linspace(0, 1, fade)
    envelope[-fade:] = np.linspace(1, 0, fade)
    tono = np.sin(2 * np.pi * frecuencia * t) * volumen * envelope
    return tono, sample_rate

def prueba_umbral(frecuencia, volumenes=[0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.70, 1.0]):
    """Reproduce el mismo tono subiendo el volumen paso a paso.
    Retorna el volumen (en %) en el que el usuario lo escuchó por primera vez."""
    print(f'\nFrecuencia: {frecuencia} Hz')
    print('El volumen irá subiendo. Presiona Enter cuando lo escuches.\n')

    for vol in volumenes:
        tono, sr = generar_tono(frecuencia, duracion=1.5, volumen=vol)
        print(f'  Volumen: {int(vol*100):>3}% ', end='', flush=True)
        display(Audio(tono, rate=sr, autoplay=True))
        time.sleep(1.8)
        respuesta = input('¿Lo escuchaste? (Enter=sí / n=no): ').strip().lower()
        if respuesta != 'n':
            print(f'  → Umbral: {int(vol*100)}%')
            return int(vol * 100)

    print('  → No detectado (pérdida significativa en esta frecuencia)')
    return 100

print('Funciones de audio listas ✓')

In [ ]:
# Frecuencias de prueba — cubren todo el espectro audible en bandas clave
# 63 Hz   = graves profundos (bombo, bajos)
# 125 Hz  = graves (voz masculina baja)
# 250 Hz  = graves medios
# 500 Hz  = medios bajos (voz humana)
# 1000 Hz = medios (referencia de audición clínica)
# 2000 Hz = medios altos (consonantes del habla)
# 4000 Hz = agudos bajos (importante para entender palabras)
# 8000 Hz = agudos (sibilantes: s, f, ch)
# 16000 Hz= agudos extremos (aire, brillo)

frecuencias_prueba = [63, 125, 250, 500, 1000, 2000, 4000, 8000, 16000]
umbrales = {}

print('PRUEBA DE AUDICIÓN — DETECCIÓN DE UMBRALES')
print('=' * 45)
print('Usa audífonos. Volumen del dispositivo al máximo.\n')

for freq in frecuencias_prueba:
    umbral = prueba_umbral(freq)
    umbrales[freq] = umbral
    print()

print('\nRESULTADOS')
print('=' * 35)
print(f'{"Frecuencia":>12}  {"Umbral":>8}  Estado')
print('-' * 35)
for freq, umb in umbrales.items():
    if umb <= 10:
        estado = '✓ Excelente'
    elif umb <= 30:
        estado = '~ Normal'
    elif umb <= 60:
        estado = '⚠ Leve pérdida'
    else:
        estado = '✗ Pérdida significativa'
    print(f'{freq:>10} Hz  {umb:>6}%  {estado}')

---
# PARTE 5 — Generar tu EQ personalizado

Aquí conectamos todo: los umbrales de tu prueba de audición se convierten en una curva de corrección, y la IA 2 genera los filtros EQ exactos para compensar tu audición.

In [ ]:
def umbrales_a_curva(umbrales, freq_cols):
    """Convierte los umbrales de la prueba a un vector de 248 frecuencias.
    Umbral alto (necesitó más volumen) = más corrección necesaria en esa banda.
    La corrección se expresa en dB: umbral 100% → -12 dB, umbral 0% → 0 dB."""
    curva = np.zeros(len(freq_cols))
    freq_prueba = list(umbrales.keys())

    for i, col in enumerate(freq_cols):
        hz = float(col.replace('f_','').replace('Hz',''))
        # Buscar la frecuencia de prueba más cercana a esta columna
        cercana = min(freq_prueba, key=lambda x: abs(x - hz))
        umbral = umbrales[cercana]
        # Mapear: umbral 0% → 0 dB (sin corrección), umbral 100% → -12 dB (máxima corrección)
        curva[i] = -((umbral / 100) * 12)

    return curva

curva_usuario = umbrales_a_curva(umbrales, freq_cols_ia2)
print('Curva auditiva generada ✓')
print(f'Bandas que necesitan corrección: {(curva_usuario < 0).sum()} de {len(freq_cols_ia2)}')
print(f'Corrección promedio necesaria: {curva_usuario.mean():.2f} dB')

In [ ]:
# Normalizar la curva del usuario con el mismo scaler de la IA 2
curva_scaled = scaler_ia2.transform(curva_usuario.reshape(1, -1))

# La IA 2 predice los 7 filtros EQ
filtros_predichos = ia2.predict(curva_scaled)[0]

print('TUS FILTROS EQ PERSONALIZADOS')
print('=' * 50)
print(f'{"#":<4} {"Tipo":<6} {"Frecuencia":>12} {"Ganancia":>10} {"Q":>8}')
print('-' * 50)
for f in range(7):
    fc   = filtros_predichos[f*3]
    gain = filtros_predichos[f*3 + 1]
    q    = filtros_predichos[f*3 + 2]
    print(f'{f+1:<4} {"PK":<6} {fc:>10.1f} Hz {gain:>+8.2f} dB {q:>8.2f}')

print()
print('Puedes copiar estos filtros en cualquier EQ paramétrico:')
print('Equalizer APO (Windows), Wavelet (Android), PowerAmp, eqMac (Mac)')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
fig.suptitle('Tu perfil auditivo y corrección EQ', fontsize=13, fontweight='bold')

# Gráfica 1: Umbrales auditivos
# Verde = excelente, naranja = normal, rojo = pérdida
freqs  = list(umbrales.keys())
umb    = list(umbrales.values())
colores = ['#e74c3c' if u > 30 else '#f39c12' if u > 10 else '#2ecc71' for u in umb]

ax1.bar([str(f) for f in freqs], umb, color=colores, edgecolor='white', linewidth=0.5)
ax1.axhline(10, color='#2ecc71', linestyle='--', linewidth=1, label='Excelente (≤10%)')
ax1.axhline(30, color='#f39c12', linestyle='--', linewidth=1, label='Normal (≤30%)')
ax1.set_ylabel('Volumen necesario para escuchar (%)')
ax1.set_xlabel('Frecuencia (Hz)')
ax1.set_title('Tu umbral auditivo por frecuencia — barras más bajas = mejor audición')
ax1.set_ylim(0, 110)
ax1.legend(fontsize=9)
for i, (f, u) in enumerate(zip(freqs, umb)):
    ax1.text(i, u + 2, f'{u}%', ha='center', fontsize=9)

# Gráfica 2: Filtros EQ
# Verde = boost (sube el volumen), rojo = cut (baja el volumen)
fc_list   = [filtros_predichos[f*3]   for f in range(7)]
gain_list = [filtros_predichos[f*3+1] for f in range(7)]
colores2  = ['#e74c3c' if g < 0 else '#2ecc71' for g in gain_list]

ax2.barh([f'{fc:.0f} Hz' for fc in fc_list], gain_list,
         color=colores2, edgecolor='white', linewidth=0.5)
ax2.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Ganancia aplicada (dB)')
ax2.set_title('Filtros EQ recomendados — cada barra es un ajuste de frecuencia')
for i, (fc, g) in enumerate(zip(fc_list, gain_list)):
    offset = 0.01 if g >= 0 else -0.01
    align  = 'left' if g >= 0 else 'right'
    ax2.text(g + offset, i, f'{g:+.2f} dB', va='center', ha=align, fontsize=9)

verde = mpatches.Patch(color='#2ecc71', label='Boost — sube esa frecuencia')
rojo  = mpatches.Patch(color='#e74c3c', label='Cut — baja esa frecuencia')
ax2.legend(handles=[verde, rojo], fontsize=9)

plt.tight_layout()
plt.savefig('perfil_auditivo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada como perfil_auditivo.png')

In [ ]:
# Guardar el perfil completo en un archivo JSON
# JSON es un formato de texto simple que puedes abrir con cualquier editor

import json
from datetime import datetime
from google.colab import files

perfil = {
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'umbrales_hz': umbrales,
    'filtros_eq': [
        {
            'filtro': f+1,
            'tipo': 'PK',
            'fc_hz':   round(filtros_predichos[f*3],   1),
            'gain_db': round(filtros_predichos[f*3+1], 2),
            'q':       round(filtros_predichos[f*3+2], 2)
        }
        for f in range(7)
    ]
}

nombre = f'eq_personalizado_{datetime.now().strftime("%Y%m%d_%H%M")}.json'
with open(nombre, 'w') as f:
    json.dump(perfil, f, indent=2, ensure_ascii=False)

files.download(nombre)
files.download('perfil_auditivo.png')
print(f'Archivos descargados: {nombre} y perfil_auditivo.png')